# External RAG Workflow
This notebook imports your exact codebase from `src` to run the active RAG workflow.

- Uses your actual `src.chunking.parent_child` logic natively.
- Directly performs upserts and retrievals via your `embed.py` and `retriever.py`.
- Leverages your remaining `semantic_cache.py` (Tier 2 Semantic Caching).
- Uses `parent_store_collection` from your Mongo database precisely as implemented.

In [1]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import from our application modules
from src.chunking.parent_child import ingest
from src.embedding.embed import upsert_chunks
from src.retrieval.retriever import search_vector_db
from src.generation.generator import generate_answer
from src.config import parent_store_collection
from src.caching.semantic_cache import get_semantic_cache, set_semantic_cache

Index 'agenticrag' already exists


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8076.68it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 1. Ingestion
Define a file to chunk, and store the parents in MongoDB (`parent_store_collection`) and children in Pinecone (`INDEX`).

In [2]:
file_to_ingest = Path(r"C:\Users\amanm\Desktop\learning\developer-chat-agent\RAGAnything.pdf") # Replace with your file path
namespace = "default_namespace" 
document_id = "doc_1"

if file_to_ingest.exists():
    print(f"Ingesting {file_to_ingest}...")
    
    # 1. Chunking (Parent-Child)
    records, parents = ingest(file_to_ingest)
    
    # 2. Store Parents in MongoDB
    print("Storing parent chunks in MongoDB...")
    for pid, text in parents.items():
        parent_store_collection.update_one(
            {"parent_id": pid, "namespace": namespace},
            {"$set": {"text": text, "document_id": document_id}},
            upsert=True
        )
        
    # 3. Embed and Upsert Children to Pinecone
    print("Upserting child chunks to Pinecone...")
    upsert_chunks(
        chunks=records, 
        namespace=namespace, 
        document_id=document_id
    )
    print("Ingestion complete!")
else:
    print(f"File {file_to_ingest} not found! Please point it to a valid document.")

Ingesting C:\Users\amanm\Desktop\learning\developer-chat-agent\RAGAnything.pdf...
Storing parent chunks in MongoDB...
Upserting child chunks to Pinecone...
493 chunks successfully inserted into namespace 'default_namespace' in index 'agenticrag'
Ingestion complete!


### 2. Retrieval & Generation with Semantic Caching
Query the dense vectors while utilizing the Tier 2 Semantic Cache.

In [3]:
query = "What is the main topic of the document?"
namespace = "default_namespace"

# 1. Check Semantic Cache (Tier 2)
cached_answer, cached_sources, query_emb = get_semantic_cache(query=query)

if cached_answer:
    print("--- SEMANTIC CACHE HIT ---")
    print(cached_answer)
else:
    print("--- CACHE MISS. RETRIEVING & GENERATING ---")
    
    # 2. Retrieve child chunks from vectors
    retrieved_chunks = search_vector_db(
        namespace=namespace, 
        query=query, 
        top_k=5
    )

    if retrieved_chunks:
        # 3. Generate Answer (Automatically uses MongoDB for Parent expansion)
        answer = generate_answer(query, retrieved_chunks, namespace)
        print("\n--- LLM ANSWER ---")
        print(answer)
        
        # 4. Save to Semantic Cache for future identical/similar queries
        sources = list(set([chunk.get("source") for chunk in retrieved_chunks]))
        set_semantic_cache(
            query=query, 
            answer=answer, 
            sources=sources,
            query_emb=query_emb
        )
    else:
        print("No relevant context found to answer the query.")

--- CACHE MISS. RETRIEVING & GENERATING ---

--- LLM ANSWER ---
The document’s primary focus is on **RAG‑Anything**, a retrieval‑augmented generation framework designed for multimodal, long‑context document understanding. It describes the benchmarks (DocBench and MMLongBench) used to evaluate the system, the challenges of multimodal document comprehension, and the baseline methods against which RAG‑Anything is compared【0】. It also outlines how the system builds visual‑layout graphs to navigate complex documents such as financial tables and multi‑panel figures【1】.  

**References**  
[0] RAGAnything.pdf, p.6.0  
[1] RAGAnything.pdf, p.6.0
2026-03-21 23:36:32 - src.caching.semantic_cache - INFO - Tier 2 (Semantic Cache) set for query
